In [3]:

# DAY 1 - SIMPLE LOCAL RAG

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import numpy as np
import torch


# ============================================
# 1. KNOWLEDGE BASE
# ============================================

documents = [
    "RAG stands for Retrieval Augmented Generation. It retrieves relevant information from documents before generating an answer.",

    "Python is a programming language commonly used for artificial intelligence, machine learning, data science and web development.",

    "Haystack is a framework used for building search and Retrieval Augmented Generation applications.",

    "FastAPI is a Python framework used to build web APIs.",

    "Docker is a platform used to package applications and their dependencies into containers.",

    "Sentence Transformers are used to convert text into numerical embeddings that can be compared for similarity."
]


# ============================================
# 2. EMBEDDING MODEL
# ============================================

embedder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded")


# ============================================
# 3. CREATE DOCUMENT EMBEDDINGS
# ============================================

document_embeddings = embedder.encode(
    documents
)

print("Documents embedded")


# ============================================
# 4. RETRIEVAL
# ============================================

def retrieve(question, top_k=2):

    question_embedding = embedder.encode(
        [question]
    )[0]

    scores = []

    for i, doc_embedding in enumerate(
        document_embeddings
    ):

        similarity = np.dot(
            question_embedding,
            doc_embedding
        ) / (
            np.linalg.norm(question_embedding)
            *
            np.linalg.norm(doc_embedding)
        )

        scores.append(
            (similarity, i)
        )

    # Highest similarity first
    scores.sort(
        reverse=True
    )

    selected = scores[:top_k]

    return [
        documents[i]
        for score, i in selected
    ]


# ============================================
# 5. LOAD LOCAL FLAN-T5 MODEL
# ============================================

model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

generator_model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name
)

print("FLAN-T5 model loaded")


# ============================================
# 6. GENERATE ANSWER
# ============================================

def generate_answer(question, context):

    context_text = "\n".join(context)

    prompt = f"""
Answer the question using only the context.

Context:
{context_text}

Question:
{question}

Answer:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    )

    with torch.no_grad():

        output = generator_model.generate(
            **inputs,
            max_new_tokens=80
        )

    answer = tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )

    return answer


# ============================================
# 7. COMPLETE RAG
# ============================================

def rag(question, top_k=2):

    context = retrieve(
        question,
        top_k
    )

    answer = generate_answer(
        question,
        context
    )

    return answer, context


# ============================================
# 8. TEST
# ============================================

question = "What is RAG?"

answer, context = rag(
    question,
    top_k=2
)


print("\n==============================")
print("QUESTION")
print("==============================")

print(question)


print("\n==============================")
print("ANSWER")
print("==============================")

print(answer)


print("\n==============================")
print("RETRIEVED CONTEXT")
print("==============================")

for i, item in enumerate(
    context,
    1
):

    print(f"{i}. {item}")


# ============================================
# 9. TOP-K EXPERIMENT
# ============================================

print("\n==============================")
print("TOP-K EXPERIMENT")
print("==============================")


for k in [1, 2, 3]:

    results = retrieve(
        question,
        top_k=k
    )

    print(f"\nTOP_K = {k}")

    for item in results:

        print("-", item)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded
Documents embedded


tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

FLAN-T5 model loaded

QUESTION
What is RAG?

ANSWER
Retrieval Augmented Generation

RETRIEVED CONTEXT
1. RAG stands for Retrieval Augmented Generation. It retrieves relevant information from documents before generating an answer.
2. FastAPI is a Python framework used to build web APIs.

TOP-K EXPERIMENT

TOP_K = 1
- RAG stands for Retrieval Augmented Generation. It retrieves relevant information from documents before generating an answer.

TOP_K = 2
- RAG stands for Retrieval Augmented Generation. It retrieves relevant information from documents before generating an answer.
- FastAPI is a Python framework used to build web APIs.

TOP_K = 3
- RAG stands for Retrieval Augmented Generation. It retrieves relevant information from documents before generating an answer.
- FastAPI is a Python framework used to build web APIs.
- Docker is a platform used to package applications and their dependencies into containers.


In [4]:
# ============================================
# DAY 2 - SIMPLE RAG EVALUATION
# ============================================

# --------------------------------------------
# 1. Evaluation questions
# --------------------------------------------

evaluation_data = [

    {
        "question": "What is RAG?",
        "expected_words": [
            "retrieval",
            "generation"
        ]
    },

    {
        "question": "What is FastAPI?",
        "expected_words": [
            "python",
            "web",
            "api"
        ]
    },

    {
        "question": "What is Docker?",
        "expected_words": [
            "package",
            "container"
        ]
    },

    {
        "question": "What is Haystack?",
        "expected_words": [
            "framework",
            "retrieval"
        ]
    }
]


# --------------------------------------------
# 2. Evaluate retrieval
# --------------------------------------------

def evaluate_retrieval(top_k):

    total = len(evaluation_data)

    correct = 0

    for item in evaluation_data:

        context = retrieve(
            item["question"],
            top_k=top_k
        )

        context_text = " ".join(
            context
        ).lower()

        found = 0

        for word in item["expected_words"]:

            if word in context_text:

                found += 1

        # If at least half of the
        # expected words are found
        if found >= len(
            item["expected_words"]
        ) / 2:

            correct += 1


    score = correct / total

    return score


# --------------------------------------------
# 3. Compare different top_k values
# --------------------------------------------

print("======================================")
print("TOP-K RETRIEVAL EVALUATION")
print("======================================")


for k in [1, 2, 3]:

    score = evaluate_retrieval(k)

    print(
        f"top_k = {k}  ->  score = {score:.2f}"
    )


# --------------------------------------------
# 4. Generate answers and inspect them
# --------------------------------------------

print("\n======================================")
print("ANSWER EVALUATION")
print("======================================")


for item in evaluation_data:

    question = item["question"]

    answer, context = rag(
        question,
        top_k=2
    )

    print("\n--------------------------------------")

    print("QUESTION:")
    print(question)

    print("\nEXPECTED KEYWORDS:")

    print(
        ", ".join(
            item["expected_words"]
        )
    )

    print("\nGENERATED ANSWER:")

    print(answer)

    print("\nRETRIEVED CONTEXT:")

    for c in context:

        print("-", c)


# --------------------------------------------
# 5. Simple answer grounding check
# --------------------------------------------

print("\n======================================")
print("GROUNDING CHECK")
print("======================================")


for item in evaluation_data:

    question = item["question"]

    answer, context = rag(
        question,
        top_k=2
    )

    context_text = " ".join(
        context
    ).lower()

    answer_words = answer.lower().split()

    matching_words = 0

    for word in answer_words:

        clean_word = word.strip(
            ".,!?;:"
        )

        if len(clean_word) > 3:

            if clean_word in context_text:

                matching_words += 1


    if len(answer_words) > 0:

        grounding_score = (
            matching_words
            /
            len(answer_words)
        )

    else:

        grounding_score = 0


    print(
        f"\n{question}"
    )

    print(
        f"Grounding score: {grounding_score:.2f}"
    )


# --------------------------------------------
# 6. Final Day 2 summary
# --------------------------------------------

print("\n======================================")
print("DAY 2 COMPLETE")
print("======================================")

print("""
✓ Created evaluation questions
✓ Created expected answers/keywords
✓ Tested retrieval quality
✓ Compared top_k = 1, 2 and 3
✓ Checked generated answers
✓ Checked answer grounding
✓ Compared retrieval configurations
""")

TOP-K RETRIEVAL EVALUATION
top_k = 1  ->  score = 1.00
top_k = 2  ->  score = 1.00
top_k = 3  ->  score = 1.00

ANSWER EVALUATION

--------------------------------------
QUESTION:
What is RAG?

EXPECTED KEYWORDS:
retrieval, generation

GENERATED ANSWER:
Retrieval Augmented Generation

RETRIEVED CONTEXT:
- RAG stands for Retrieval Augmented Generation. It retrieves relevant information from documents before generating an answer.
- FastAPI is a Python framework used to build web APIs.

--------------------------------------
QUESTION:
What is FastAPI?

EXPECTED KEYWORDS:
python, web, api

GENERATED ANSWER:
a Python framework used to build web APIs

RETRIEVED CONTEXT:
- FastAPI is a Python framework used to build web APIs.
- RAG stands for Retrieval Augmented Generation. It retrieves relevant information from documents before generating an answer.

--------------------------------------
QUESTION:
What is Docker?

EXPECTED KEYWORDS:
package, container

GENERATED ANSWER:
platform used to pac